# 03 — Statistical Tests

Move beyond visual patterns to test whether observed differences are statistically significant. We use non-parametric tests throughout — appropriate for our skewed, bounded distributions.

In [ ]:
from pathlib import Path
import pandas as pd
from src.data.load import load_foodhub
from src.features.build import clean_data, engineer_features
from src.analysis.stats import (
    compare_weekday_weekend,
    compare_independence,
    compare_across_groups,
    run_all_tests,
)
from src.visualization.plots import plot_test_result, plot_effect_sizes

In [ ]:
df = load_foodhub(Path("data/raw/foodhub_order.csv"))
df = clean_data(df)
df = engineer_features(df)

## Weekday vs Weekend: Order Cost

Do customers spend more on weekends?

In [ ]:
result = compare_weekday_weekend(df, "cost_of_the_order")
for k, v in result.items():
    print(f"{k}: {v}")

In [ ]:
weekday = df.loc[df["day_of_the_week"] == "Weekday", "cost_of_the_order"]
weekend = df.loc[df["day_of_the_week"] == "Weekend", "cost_of_the_order"]
fig = plot_test_result(weekday, weekend, "Weekday", "Weekend", "Order Cost ($)", result["p_value"])
fig

## Weekday vs Weekend: Delivery Time

In [ ]:
result = compare_weekday_weekend(df, "delivery_time")
for k, v in result.items():
    print(f"{k}: {v}")

## Cuisine Preferences by Day

Do people order different cuisines on weekends vs weekdays? (Chi-square test)

In [ ]:
result = compare_independence(df, "cuisine_type", "day_of_the_week")
for k, v in result.items():
    print(f"{k}: {v}")

## Rating Behavior by Day

Are customers more likely to leave a rating on weekdays or weekends?

In [ ]:
result = compare_independence(df, "day_of_the_week", "has_rating")
for k, v in result.items():
    print(f"{k}: {v}")

## Cost Across Cuisines

Do different cuisines have significantly different price points? (Kruskal-Wallis)

In [ ]:
result = compare_across_groups(df, "cuisine_type", "cost_of_the_order")
for k, v in result.items():
    print(f"{k}: {v}")

## All Tests Summary

In [ ]:
results = run_all_tests(df)
summary = pd.DataFrame(results)[["name", "test", "p_value", "effect_size", "interpretation"]]
summary

## Effect Sizes

In [ ]:
effect_dict = {r["name"]: abs(r["effect_size"]) for r in results}
fig = plot_effect_sizes(effect_dict)
fig

## Key Takeaways

- Most weekday vs weekend differences show **small effect sizes** even when statistically significant — practically meaningful differences are limited
- Cuisine type has the strongest influence on cost (expected) and delivery time
- Rating behavior is largely independent of day of week
- The small effect sizes suggest that day-of-week is not a strong driver of operational differences